# Chapter 22 — Packages: Shipping an Agent

**Companion to *Pi Agents*** · [`Pi Agents` chapter 22](https://programmer.ie/books/pi/22-chapter/)

| | |
|---|---|
| Chapter | `22-chapter.md` · weight 22 · `/books/pi/22-chapter/` |
| Notebook | `notebooks/pi/22-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Does `pi install ./pkg` record a relative path and copy nothing — and when
is the host-dependency warning emitted?

The chapter's point: a package is the distribution unit for extensions, skills,
prompts and themes. `pi install` records a relative path in settings.json and
copies nothing. Filters narrow what loads; they cannot expose what the package
did not declare. Host packages belong in `peerDependencies` with a `*` range,
not `dependencies`.


## What this notebook establishes

- `pi install ./pkg` records a **relative path** from the agent directory in `settings.json` — nothing is copied into the agent directory.
- `pi list` shows the installed package under 'User packages:'.
- A package's prompt template and extension load from conventional directories with no manifest needed.
- `pi -e` loads a package for one invocation and **writes nothing to settings**.
- `pi remove` removes the package from settings and it no longer loads.
- The **object form** in settings filters a package: `prompts: []` excludes prompts but leaves extensions and skills.
- A filter **narrows** what the package declares; it **cannot expose** something the package did not declare in its manifest.
- The **same package listed twice** (as string and object) loads once — identity is the resolved path.
- An explicit `pi` manifest in `package.json` **replaces** conventional discovery — only declared resources load.
- `--local` writes to the project's `.pi/settings.json`, which loads **only after the project is trusted**.
- A host-provided package in `dependencies` draws a **warning**; in `peerDependencies` with a `*` range it does not.

## What this notebook does **not** establish

- **No npm or git source** — every source is a local path and every run is offline; no published package is exercised end to end (`ch22-lim1`).
- **No skill theme loading** is demonstrated — the tests focus on prompts and extensions.
- **The `pi` manifest's `mcpServers` field** is not exercised here.
- **The package gallery** (`pi.image`, `pi.video`) is documented, not run.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Run** — the shipped binary (`pi`) offline: `install`, `list`, `remove`, `-e`, `--local` under `--approve`/`--no-approve`.
* **Evidence scope:** `shipped_binary`. The package mechanics are the real Pi CLI.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(22))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/22-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: what `pi install` writes, and what it does not do

In [3]:
PKG = pinb.driver_source("ch22-install.ts")
pkg = pinb.driver(PKG, label="ch22-install", timeout=300)
print(pinb.md_table(
    ["check", "result"],
    [["exit code", "0" if pkg["install"]["exit"] == 0 else "non-zero"],
     ["entry is string", "yes" if pkg["install"]["entryIsString"] else "no"],
     ["source recorded", pkg["install"]["source"]],
     ["is absolute path", "yes" if pkg["install"]["isAbsolute"] else "no"],
     ["relative to settings", pkg["install"]["relativeToSettings"]],
     ["resolves to package dir", "yes" if pkg["install"]["resolvesToPackage"] else "no"],
     ["package still on disk", "yes" if pkg["install"]["packageStillOnDisk"] else "no"],
     ["copied into agent dir", "yes" if pkg["install"]["copiedIntoAgentDir"] else "no"],
     ["agent dir entries", ", ".join(pkg["install"]["agentDirEntries"])]],
))

| check | result |
|---|---|
| exit code | 0 |
| entry is string | yes |
| source recorded | ..\my-pkg |
| is absolute path | no |
| relative to settings | ../my-pkg |
| resolves to package dir | yes |
| package still on disk | yes |
| copied into agent dir | no |
| agent dir entries | auth.json, models-store.json, settings.json |


In [4]:
pinb.show_checks([
    pinb.check("install exits 0", pkg["install"]["exit"] == 0, "exit 0"),
    pinb.check("entry is a relative path from agent dir", pkg["install"]["relativeToSettings"] == "../my-pkg", "correct relative path"),
    pinb.check("resolves back to the package directory", pkg["install"]["resolvesToPackage"], "resolves correctly"),
    pinb.check("package NOT copied into agent dir", not pkg["install"]["copiedIntoAgentDir"], "no copy"),
    pinb.check("package still on disk at original location", pkg["install"]["packageStillOnDisk"], "original intact"),
])

== Assertions ==
  assertion  observed
----  ------------------------------------------  ---------------------
PASS  install exits 0  exit 0
PASS  entry is a relative path from agent dir  correct relative path
PASS  resolves back to the package directory  resolves correctly
PASS  package NOT copied into agent dir  no copy
PASS  package still on disk at original location  original intact

5/5 assertions held.


## Listing and one-shot invocation

In [5]:
print(pinb.md_table(
    ["check", "result"],
    [["list shows package", "yes" if pkg["list"]["showsPackage"] else "no"],
     ["list has User packages section", "yes" if pkg["list"]["section"] else "no"],
     ["one-shot expands template", "yes" if pkg["oneShot"]["expanded"] else "no"],
     ["one-shot writes to settings", "yes" if pkg["oneShot"]["settingsWritten"] else "no"]],
))

| check | result |
|---|---|
| list shows package | yes |
| list has User packages section | yes |
| one-shot expands template | yes |
| one-shot writes to settings | no |


In [6]:
pinb.show_checks([
    pinb.check("pi list shows the package", pkg["list"]["showsPackage"], "listed"),
    pinb.check("pi -e expands the template", pkg["oneShot"]["expanded"], "expanded"),
    pinb.check("pi -e does NOT write to settings", not pkg["oneShot"]["settingsWritten"], "no settings write"),
])

== Assertions ==
  assertion  observed
----  --------------------------------  -----------------
PASS  pi list shows the package  listed
PASS  pi -e expands the template  expanded
PASS  pi -e does NOT write to settings  no settings write

3/3 assertions held.


## Removal

In [7]:
print(pinb.md_table(
    ["check", "result"],
    [["remove exits 0", "yes" if pkg["remove"]["exit"] == 0 else "no"],
     ["package no longer listed", "no" if pkg["remove"]["stillListed"] else "yes"],
     ["template no longer expands", "no" if pkg["remove"]["stillExpands"] else "yes"]],
))

| check | result |
|---|---|
| remove exits 0 | yes |
| package no longer listed | yes |
| template no longer expands | yes |


In [8]:
pinb.show_checks([
    pinb.check("remove exits 0", pkg["remove"]["exit"] == 0, "exit 0"),
    pinb.check("package removed from list", not pkg["remove"]["stillListed"], "unlisted"),
    pinb.check("template no longer expands after remove", not pkg["remove"]["stillExpands"], "unloaded"),
])

== Assertions ==
  assertion  observed
----  ---------------------------------------  --------
PASS  remove exits 0  exit 0
PASS  package removed from list  unlisted
PASS  template no longer expands after remove  unloaded

3/3 assertions held.


## The chapter's own tests

All 11 tests from the chapter's test file.

## The chapter's canonical evidence

These are the chapter's own tests, executed unmodified through Node's test
runner. Every ledger row the chapter quotes comes from a named case in this
file, so running it here is running the chapter's evidence, not a copy of it.

In [9]:
PATTERNS = 'ch22-packages/packages.test.ts'
tap = pinb.run_tests(PATTERNS, timeout=300)
pinb.show_tap(tap, files='ch22-packages/packages.test.ts', show="package")

$ node --test ch22-packages/packages.test.ts
  PASS  11 passed, 0 failed, 0 skipped  in 7328 ms

  tests matching 'package':
    ok   an installed package's prompt template and extension load, with no manifest, from the conventional directories
    ok   pi -e loads a package for one invocation and writes nothing to settings
    ok   the object form filters a package: [] loads none of that type, and the other types still load
    ok   filters narrow what the package declares; they cannot expose something it did not
    ok   the same package listed twice loads once: identity is the resolved path
    ok   an explicit manifest can point at resources elsewhere in the package
    ok   a host-provided package listed in dependencies draws a warning; in peerDependencies with a * range it does not


## Your turn: predict, then run

**Predict first.** What happens if you `pi install` the same package twice? Does
it appear twice in `pi list`?

**Then change one input.** The object form `{ source: "...", prompts: [] }`
filters out prompts. What does `{ source: "...", extensions: [] }` do?

**Predict the boundary.** The chapter says "filters narrow what the package
declares; they cannot expose what the package did not declare." What if the
package declares `prompts: ["./elsewhere/*.md"]` and you filter
`prompts: ["prompts/greet.md"]` — does the filter expose the undeclared
`prompts/greet.md`?

In [10]:
print("Predictions:")
print("  1. Install twice: appears once in list (settings.json deduplicates).")
print("  2. extensions: [] filters out extensions but leaves prompts/skills.")
print("  3. Filter cannot expose undeclared: prompts/greet.md stays hidden.")
print()
print("Observed above:")
print(f"  install: relative={pkg["install"]["relativeToSettings"]}, copied={pkg["install"]["copiedIntoAgentDir"]}")
print(f"  list: shows={pkg["list"]["showsPackage"]}")
print(f"  one-shot: expanded={pkg["oneShot"]["expanded"]}, settings_written={pkg["oneShot"]["settingsWritten"]}")
print(f"  remove: unlisted={not pkg["remove"]["stillListed"]}, unexpands={not pkg["remove"]["stillExpands"]}")
print()
assert not pkg["install"]["copiedIntoAgentDir"]
assert pkg["install"]["relativeToSettings"] == "../my-pkg"
assert pkg["list"]["showsPackage"]
assert pkg["oneShot"]["expanded"] and not pkg["oneShot"]["settingsWritten"]
assert not pkg["remove"]["stillListed"] and not pkg["remove"]["stillExpands"]
print("held: install records relative path, copies nothing; remove unloads; -e is one-shot")

Predictions:
  1. Install twice: appears once in list (settings.json deduplicates).
  2. extensions: [] filters out extensions but leaves prompts/skills.
  3. Filter cannot expose undeclared: prompts/greet.md stays hidden.

Observed above:
  install: relative=../my-pkg, copied=False
  list: shows=True
  one-shot: expanded=True, settings_written=False
  remove: unlisted=True, unexpands=True

held: install records relative path, copies nothing; remove unloads; -e is one-shot


## Interpretation

A Pi package is the distribution unit for your agent customizations. The key
mechanics and their boundaries:

| Mechanism | What it does | The boundary |
|---|---|---|
| `pi install ./pkg` | Records relative path in `settings.json` | **Copies nothing** — the package stays where it is |
| `pi list` | Shows installed packages | User packages vs project packages |
| `pi -e ./pkg` | One-shot load | **Writes nothing to settings** |
| `pi remove` | Removes from settings | Package no longer loads |
| Settings object form | Filters by resource type | `prompts: []` excludes prompts; `skills: ["a"]` keeps only `a` |
| Package `pi` manifest | Declares resources explicitly | **Replaces** conventional discovery — undeclared resources don't load |
| `pi install --local` | Writes to `.pi/settings.json` | **Requires trust** — project must be approved |

The dependency rule:
* `dependencies: { "@earendil-works/pi-ai": "*" }` → **warning** (host provides it)
* `peerDependencies: { "@earendil-works/pi-ai": "*" }` → **no warning**

The filter rule:
* Filters **narrow** what the package declares
* Filters **cannot expose** what the package did not declare
* Identity is the **resolved path** — same package twice = loads once

Practical rules:
1. **Use relative local paths** — `pi install ./my-pkg` records `../my-pkg`, works across machines.
2. **Don't bundle host packages** — declare them in `peerDependencies` with `*`.
3. **Use the object form to filter** — keep what you want, exclude what you don't.
4. **`--local` is for project-specific packages** — they require trust approval.

## Sources, execution mode and limitations

**Execution mode:** **Run** — the shipped binary (`pi`) offline: `install`, `list`, `remove`, `-e`, `--local` under `--approve`/`--no-approve`.

**Evidence scope:** `shipped_binary`. The package mechanics are the real Pi CLI.

### What was read or run

- **Ran** `drivers/ch22-install.ts`: real package directory installed with shipped binary.
- **Ran** `drivers/ch22-scope.ts`: `--local` trust gating and identity deduplication.
- **Ran** `drivers/ch22-depends.ts`: dependency warning and filter narrowing/exposure rules.
- **Ran** `ch22-packages/packages.test.ts` (11 tests).
- **Read** `examples/evidence.json`, chapter 22 rows (11 claims).

### Limitations

- **No npm or git source** — every source is a local path and every run is offline; no published package is exercised end to end (`ch22-lim1`).
- **No skill theme loading** is demonstrated — the tests focus on prompts and extensions.
- **The `pi` manifest's `mcpServers` field** is not exercised here.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
